# Basic Usage

For plotting images with widgets and callbacks for interactivity.

In [ ]:
import numpy as np
import panel as pn

from libertem_ui.figure import ApertureFigure
from libertem_ui.display.display_base import Cursor

## Overview

`LiberTEM-panel-ui` is based on two libraries:

- `bokeh`, which provides a fairly low-level plotting interface (figure, axes, glyphs etc) and infrastructure for interactivity in a browser
- `panel`, which itself is based on `bokeh`, which provides a higher-level interface for widgets, callbacks and easy display of `bokeh` components in Jupyter Notebooks as well as in a browser. It also provides a good set of template, themeing and layout infrastrucure, as well as advanced widgets like code editors, 3D display or interactive tools for tabular data.

`LiberTEM-panel-ui` provides two main features:

- `libertem_ui.figure.ApertureFigure`, which displays various sorts of *image-like* data on a `bokeh.plotting.Figure`, and provides some useful convenience features on this figure, such as adjustable colormaps, automatic downsampling, and mask / ROI tools
- Under `libertem_ui.display.display_base`, various wrappers for `bokeh.models.Glyph`, i.e. things which can be plotted on a figure axis like a set of points, polygons, lines etc. The added value is:
  - Simplified constructors
  - Unified creation of the data source, the glyph(s), and adding them to one-or-more figures, to reduce boilerplate
  - Common interface for updating data
  - Easy creation of tools (such as click to create a point)
  - Structure to handle groups of `Glyph` as one object which share the same data source and methods

Otherwise, the features are quite "thin", and there is no attempt to re-define the styling or callback infrastructre of `bokeh` or `panel`. If you want to modify a property of the axes, for example, you do it via the `bokeh` API on the `ap_fig.fig` property, for example:

```python
ap_fig.fig.xaxis.axis_label = "Width"
```

If you want to trigger a callback when the data of a figure changes, you do it directly through `bokeh` with `data_source.on_change(...)`, for example.

## Simple display and callbacks

To display an image on a new figure (or stack, complex image etc) create a new `ApertureFigure`. To cause it to display after a notebook cell the `fig.layout` property must be returned from that cell. `.layout` is a `panel.layouts.Column` layout which contains the figure and its support widgets. The cell that defines the figure and the cell which displays it need not be the same.

In [ ]:
fig1 = ApertureFigure.new(np.random.uniform(size=(32, 64)), title=f"A random image")
fig1.layout

We can update the image data by defining a widget to trigger a callback, then calling the figure's `update` method:

In [ ]:
im_shape = (32, 64)
fig2 = ApertureFigure.new(np.random.uniform(size=im_shape), title=f"An updating image")

def update_image(event):
    fig2.update(np.random.uniform(size=im_shape))

btn = pn.widgets.Button(name="Update figure")
btn.on_click(update_image)
# Since we already have a Column layout for the figure, use it rather than creating our own
fig2.layout.insert(0, btn)
fig2.layout

A `Button` has a simplified `on_click` callback endpoint, more general widgets use `widget.param.watch(cb, "property")`, here we use a slider to update the figure title:

In [ ]:
slider = pn.widgets.FloatSlider(name="Figure title", start=0., end=1., step=0.1, value=0.)
fig3 = ApertureFigure.new(np.random.uniform(size=(32, 64)), title=f"Slider value: {slider.value}")

def update_title(event):
    fig3.fig.title.text = f"Slider value: {event.new}"

slider.param.watch(update_title, "value_throttled")
fig3.layout.insert(0, slider)
fig3.layout

## Callback from changing `Glyph` data

Here we add a `libertem_ui.display.display_base.Cursor` to the figure, and update some text in response to the `Cursor` being dragged on screen.

Behind every `Glyph` is a `bokeh` `ColumnDataSource`, which is a wrapper of a dict-like structure with equal-length columns. The `Glyph` combines with the `ColumnDataSource` as a `Renderer` on a figure. If the rendered glyph is updated on screen, the change is synchronised back to Python, and if the `ColumnDataSource` is updated on the Python side, the change is synchronised to the displayed renderer. This update can trigger callbacks both in the browser as Javascript, or in Python.

A `Cursor` (and other components in `libertem_ui.display`) is a wrapper around `Glyph(s)`, a `ColumDataSource` and one-or-more `Renderers` for one-or-more figures. The data source is always available under `cursor.cds` so that callbacks can be registered.

The `Glyph` itself sets visual properties of something to be rendered, e.g. colour. It is usually accessible under `cursor.glyph`, though this interface needs to be unified.

In [ ]:
fig4 = ApertureFigure.new(np.arange(64 * 64).reshape(64, 64), title=f"Has a cursor")
# Add a "Cursor" to the figure on top of the image
# which is a single point with an X/Y position and convenience methods
cursor = (
    Cursor
    .new()
    .from_pos(x=32, y=32)
    .on(fig4.fig)
    .editable(selected=True)  # this adds the necessary components to make the cursor draggable
)
cursor.glyph.line_color = "red"
title_str = lambda cur: f"Cursor position: x={cur.current_pos().x:.1f}, y={cur.current_pos().y:.1f}"
label = pn.widgets.StaticText(value=title_str(cursor))

def update_label(attr, old, new):
    # This is a "bokeh-style" callback because it will trigger directly from a ColumnDataSource
    # The callback must have three arguments [attr, old, new]
    # attr will be "data"
    # old will be the CDS dict before the trigger
    # new will be the CDS dict after the trigger
    label.value = title_str(cursor)

# trigger whenever the "data" attribute of the cursor ColumnDataSource changes
cursor.cds.on_change("data", update_label)

fig4.layout.insert(0, label)
fig4.layout

## Use as a UDF Live Plot

We can also use the `ApertureFigure` as a UDF Live Plot, through the specialised subclass `libertem_ui.live_plot.AperturePlot`.

In [ ]:
import libertem.api as lt
from libertem.udf.sumsigudf import SumSigUDF
from libertem.udf.sum import SumUDF
from libertem_ui.live_plot import AperturePlot

In [ ]:
ctx = lt.Context.make_with("inline")
ds_shape = (5, 7, 8, 8)
ds = ctx.load("memory", data=np.arange(np.prod(ds_shape)).reshape(ds_shape), num_partitions=4)
ds.shape

Define a convenience UDF to show sequential updating partition-by-partition:

In [ ]:
import time

class GoSlowUDF(SumUDF):
    def process_tile(self, tile):
        time.sleep(0.25)
        return super().process_tile(tile)

As is common in notebooks, we need to declare and display the UI before actually running the computation. This also demonstrates how to layout multiple figures in the same cell output:

In [ ]:
sumsig_udf = SumSigUDF()
sumsig_plot = AperturePlot.new(ds, sumsig_udf, title="Sum-over-sig")
sum_udf = SumUDF()
sum_plot = AperturePlot.new(ds, sum_udf, title="Sum-over-nav")
pn.layout.Row(
    sumsig_plot.layout,
    sum_plot.layout,
)


Once displayed, we can run the computation itself

In [ ]:
_ = ctx.run_udf(ds, [GoSlowUDF(), sumsig_udf, sum_udf], plots=[sumsig_plot, sum_plot])